In [1]:

# ogbn-arxiv _minibatch_baseline with Early Stopping


import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import random
from torch_geometric.nn import GCNConv
from torch_geometric.loader import NeighborLoader
import torch_geometric.transforms as T
from ogb.nodeproppred import PygNodePropPredDataset, Evaluator

# ==========================================
# 实验配置
# ==========================================
class BaseConfig:
    hidden_dim = 128
    heads = 8 
    total_hidden = 1024 
    epochs = 300        
    patience = 40       
    lr_init = 0.001
    batch_size = 512
    weight_decay = 1e-4
    seeds = [1,7,27,1227,314] # 按需修改种子列表

# ==========================================
# 增强型 GCN 模型 
# ==========================================
class GCN_Baseline(torch.nn.Module):
    def __init__(self, in_size, hidden_size, out_size):
        super().__init__()
        self.conv1 = GCNConv(in_size, hidden_size)
        self.bn1 = nn.BatchNorm1d(hidden_size) # 【新增】BN 层 1
        
        self.conv2 = GCNConv(hidden_size, hidden_size)
        self.bn2 = nn.BatchNorm1d(hidden_size) # 【新增】BN 层 2
        
        # 为了严谨对比 NRS 的分类器结构，这里也采用类似的 Linear 布局
        self.post_lin = nn.Linear(hidden_size, out_size)
        self.dropout = nn.Dropout(0.3)

    def forward(self, x, edge_index):
        # 第一层：Conv -> BN -> ReLU -> Dropout
        x = self.conv1(x, edge_index)
        x = self.bn1(x)
        x = F.relu(x)
        x = self.dropout(x)
        
        # 第二层：Conv -> BN -> ReLU
        x = self.conv2(x, edge_index)
        x = self.bn2(x)
        x = F.relu(x)
        
        # 输出层
        x = self.post_lin(x)
        return F.log_softmax(x, dim=1)

# ==========================================
# 实验运行器
# ==========================================
def run_experiment(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    dataset = PygNodePropPredDataset(name='ogbn-arxiv', root='data', transform=T.ToUndirected())
    data = dataset[0].to(device)
    split_idx = dataset.get_idx_split()
    
    train_loader = NeighborLoader(
        data, num_neighbors=[15, 10], batch_size=BaseConfig.batch_size,
        input_nodes=split_idx['train'], shuffle=True, num_workers=4
    )

    model = GCN_Baseline(dataset.num_features, BaseConfig.total_hidden, dataset.num_classes).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=BaseConfig.lr_init, weight_decay=BaseConfig.weight_decay)
    evaluator = Evaluator(name='ogbn-arxiv')

    best_val_acc = 0
    final_test_acc = 0
    stagnant_count = 0  

    for epoch in range(1, BaseConfig.epochs + 1):
        model.train()
        total_loss = 0
        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            out = model(batch.x, batch.edge_index)
            # 损失只计算目标节点（batch_size 以前的部分）
            loss = F.nll_loss(out[:batch.batch_size], batch.y.squeeze()[:batch.batch_size])
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        if epoch % 5 == 0:
            model.eval()
            with torch.no_grad():
                out = model(data.x, data.edge_index)
                y_pred = out.argmax(dim=-1, keepdim=True)
                
                val_acc = evaluator.eval({'y_true': data.y[split_idx['valid']], 'y_pred': y_pred[split_idx['valid']]})['acc']
                test_acc = evaluator.eval({'y_true': data.y[split_idx['test']], 'y_pred': y_pred[split_idx['test']]})['acc']
                
                if val_acc > best_val_acc:
                    best_val_acc = val_acc
                    final_test_acc = test_acc
                    stagnant_count = 0  
                    status = "[*] NEW BEST"
                else:
                    stagnant_count += 5 
                    status = "[ ]"
                
                print(f"Seed: {seed} | Ep: {epoch:03d} | Loss: {total_loss/len(train_loader):.4f} | Val: {val_acc:.4f} | Test: {test_acc:.4f} {status}")

                if stagnant_count >= BaseConfig.patience:
                    print(f"Early Stopping triggered at epoch {epoch}.")
                    break

    return final_test_acc

if __name__ == "__main__":
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    results = []
    
    print(f"Starting GCN+BN Baseline Experiment on ogbn-arxiv...")
    for s in BaseConfig.seeds:
        test_acc = run_experiment(s)
        results.append(test_acc)
        print(f"==> Seed {s} Finished. Best Test Acc: {test_acc:.4f}\n")

    mean_acc = np.mean(results)
    std_acc = np.std(results)
    
    print("-" * 30)
    print(f"GCN+BN Final Results ({len(BaseConfig.seeds)} Seeds):")
    print(f"Mean Test Accuracy: {mean_acc:.4f}")
    print(f"Std Deviation: {std_acc:.4f}")
    print(f"Latex Format: {mean_acc*100:.2f} \pm {std_acc*100:.2f}")
    print("-" * 30)

/home/malina/.venv/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Starting GCN+BN Baseline Experiment on ogbn-arxiv...


/home/malina/.venv/lib/python3.8/site-packages/ogb/nodeproppred/dataset_pyg.py:69: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.data, self.slices = torch.load(self.pro

Seed: 1 | Ep: 005 | Loss: 0.8631 | Val: 0.6486 | Test: 0.6456 [*] NEW BEST
Seed: 1 | Ep: 010 | Loss: 0.7877 | Val: 0.6844 | Test: 0.6799 [*] NEW BEST
Seed: 1 | Ep: 015 | Loss: 0.7323 | Val: 0.6964 | Test: 0.6921 [*] NEW BEST
Seed: 1 | Ep: 020 | Loss: 0.6912 | Val: 0.6749 | Test: 0.6722 [ ]
Seed: 1 | Ep: 025 | Loss: 0.6532 | Val: 0.6626 | Test: 0.6475 [ ]
Seed: 1 | Ep: 030 | Loss: 0.6219 | Val: 0.6674 | Test: 0.6553 [ ]
Seed: 1 | Ep: 035 | Loss: 0.5977 | Val: 0.6828 | Test: 0.6744 [ ]
Seed: 1 | Ep: 040 | Loss: 0.5728 | Val: 0.6718 | Test: 0.6554 [ ]
Seed: 1 | Ep: 045 | Loss: 0.5566 | Val: 0.6737 | Test: 0.6628 [ ]
Seed: 1 | Ep: 050 | Loss: 0.5384 | Val: 0.6829 | Test: 0.6651 [ ]
Seed: 1 | Ep: 055 | Loss: 0.5212 | Val: 0.6824 | Test: 0.6752 [ ]
Early Stopping triggered at epoch 55.
==> Seed 1 Finished. Best Test Acc: 0.6921



/home/malina/.venv/lib/python3.8/site-packages/ogb/nodeproppred/dataset_pyg.py:69: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.data, self.slices = torch.load(self.pro

Seed: 7 | Ep: 005 | Loss: 0.8681 | Val: 0.6793 | Test: 0.6806 [*] NEW BEST
Seed: 7 | Ep: 010 | Loss: 0.7917 | Val: 0.6657 | Test: 0.6388 [ ]
Seed: 7 | Ep: 015 | Loss: 0.7364 | Val: 0.6877 | Test: 0.6832 [*] NEW BEST


KeyboardInterrupt: 

In [1]:
# 1.31 版本 baseline

# ogbn-arxiv GCN-Baseline (Aligned with Task-SPD/PANS) - 8 Seeds Version

import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import random
from torch_geometric.nn import GCNConv
from torch_geometric.loader import NeighborLoader
import torch_geometric.transforms as T
from ogb.nodeproppred import PygNodePropPredDataset, Evaluator

# ==========================================
# 1. 实验配置
# ==========================================
class BaseConfig:
    hidden_dim = 128
    total_hidden = 1024  # 对齐 NRS 的 hidden_dim * 8
    epochs = 150
    patience = 30        # 连续 6 次评估（30轮）不上升则早停
    lr_init = 0.001
    batch_size = 512
    weight_decay = 5e-4
    # 扩展至 8种子，涵盖了常用的质数和随机分布
    #seeds = [1, 7, 27, 42, 100, 314, 1227, 2026] 
    seeds = [1,7,314,1227,2026]

# ==========================================
# 2. 对齐型 GCN 模型 
# ==========================================
class GCN_Aligned(torch.nn.Module):
    def __init__(self, in_size, hidden_size, out_size):
        super().__init__()
        # 1. 标签嵌入层 (与 PANS 对齐)
        self.label_emb = nn.Embedding(out_size + 1, hidden_size)
        self.res_lin = nn.Linear(in_size, hidden_size)
        
        # 2. GCN 层 (对齐 NRS 的两层感知结构)
        self.conv1 = GCNConv(hidden_size, hidden_size)
        self.bn1 = nn.BatchNorm1d(hidden_size)
        self.conv2 = GCNConv(hidden_size, hidden_size)
        self.bn2 = nn.BatchNorm1d(hidden_size)
        
        # 3. 分类器 (严格对齐 PANS 的 Sequential 结构)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.BatchNorm1d(hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_size // 2, out_size)
        )

    def forward(self, x, edge_index, y_label, target_mask):
        # 信号融合 (对齐 NRS 的语义感知层)
        x_proj = self.res_lin(x)
        y_embedded = self.label_emb(y_label)
        x = x_proj + y_embedded
        
        # Conv 1
        x = F.relu(self.bn1(self.conv1(x, edge_index)))
        # Conv 2 + 残差链接
        x = F.relu(self.bn2(self.conv2(x, edge_index))) + x_proj
        
        logits = self.classifier(x)
        return F.log_softmax(logits, dim=1)

# ==========================================
# 3. Mini-batch 评估函数 
# ==========================================
@torch.no_grad()
def inference_minibatch(model, loader, device, evaluator, num_classes):
    model.eval()
    y_preds, y_trues = [], []
    for batch in loader:
        batch = batch.to(device)
        y_input = batch.y.squeeze().clone()
        y_input[:batch.batch_size] = num_classes  # 标签隔离：抹除中心节点标签
        
        mask = torch.zeros(batch.num_nodes, dtype=torch.bool, device=device)
        mask[:batch.batch_size] = True
        
        out = model(batch.x, batch.edge_index, y_input, mask)
        y_preds.append(out[:batch.batch_size].argmax(dim=-1).cpu())
        y_trues.append(batch.y[:batch.batch_size].cpu())
        
    y_pred = torch.cat(y_preds, dim=0).numpy().reshape(-1, 1)
    y_true = torch.cat(y_trues, dim=0).numpy().reshape(-1, 1)
    return evaluator.eval({'y_true': y_true, 'y_pred': y_pred})['acc']

# ==========================================
# 4. 实验运行器
# ==========================================
def run_experiment(seed, device):
    # 严格控制随机性
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    
    dataset = PygNodePropPredDataset(name='ogbn-arxiv', root='data', transform=T.ToUndirected())
    data = dataset[0]
    split_idx = dataset.get_idx_split()
    evaluator = Evaluator(name='ogbn-arxiv')

    train_loader = NeighborLoader(data, num_neighbors=[20, 15], batch_size=BaseConfig.batch_size,
                                 input_nodes=split_idx['train'], shuffle=True)
    valid_loader = NeighborLoader(data, num_neighbors=[-1, -1], batch_size=BaseConfig.batch_size,
                                 input_nodes=split_idx['valid'], shuffle=False)
    test_loader = NeighborLoader(data, num_neighbors=[-1, -1], batch_size=BaseConfig.batch_size,
                                input_nodes=split_idx['test'], shuffle=False)

    model = GCN_Aligned(dataset.num_features, BaseConfig.total_hidden, dataset.num_classes).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=BaseConfig.lr_init, weight_decay=BaseConfig.weight_decay)

    best_val_acc, final_test_acc, stagnant_count = 0, 0, 0

    for epoch in range(1, BaseConfig.epochs + 1):
        model.train()
        total_loss = 0
        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            
            y_input = batch.y.squeeze().clone()
            y_input[:batch.batch_size] = dataset.num_classes # Label Reuse 对齐
            
            out = model(batch.x, batch.edge_index, y_input, None)
            loss = F.nll_loss(out[:batch.batch_size], batch.y.squeeze()[:batch.batch_size])
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        if epoch % 5 == 0:
            val_acc = inference_minibatch(model, valid_loader, device, evaluator, dataset.num_classes)
            test_acc = inference_minibatch(model, test_loader, device, evaluator, dataset.num_classes)
            
            if val_acc > best_val_acc:
                best_val_acc, final_test_acc, stagnant_count = val_acc, test_acc, 0
                status = "[*] NEW BEST"
            else:
                stagnant_count += 5
                status = "[ ]"
            
            print(f"Seed: {seed:4d} | Ep: {epoch:03d} | Loss: {total_loss/len(train_loader):.4f} | Val: {val_acc:.4f} | Test: {test_acc:.4f} {status}")
            if stagnant_count >= BaseConfig.patience: 
                print(f"Early stop seed {seed} at epoch {epoch}")
                break
            
    return final_test_acc

if __name__ == "__main__":
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    results = []
    
    print("-" * 50)
    print(f"Starting Baseline: Aligned GCN + Label Reuse ({len(BaseConfig.seeds)} Seeds)")
    print("-" * 50)

    for i, s in enumerate(BaseConfig.seeds):
        print(f"\n[Run {i+1}/{len(BaseConfig.seeds)}] Seed: {s}")
        acc = run_experiment(s, device)
        results.append(acc)
    
    # 统计计算
    mean_res = np.mean(results) * 100
    std_res = np.std(results) * 100
    
    print("\n" + "=" * 50)
    print("FINAL BASELINE RESULTS")
    print("-" * 50)
    print(f"Seeds tested: {BaseConfig.seeds}")
    print(f"All Test Acc: {[f'{a:.4f}' for a in results]}")
    print(f"Mean Accuracy: {mean_res:.2f}%")
    print(f"Std Deviation: {std_res:.2f}%")
    print(f"LaTeX Format: {mean_res:.2f} \pm {std_res:.2f}")
    print("=" * 50)


/home/malina/.venv/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


--------------------------------------------------
Starting Baseline: Aligned GCN + Label Reuse (5 Seeds)
--------------------------------------------------

[Run 1/5] Seed: 1


/home/malina/.venv/lib/python3.8/site-packages/ogb/nodeproppred/dataset_pyg.py:69: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.data, self.slices = torch.load(self.pro

Seed:    1 | Ep: 005 | Loss: 0.8404 | Val: 0.7474 | Test: 0.7311 [*] NEW BEST


KeyboardInterrupt: 

In [ ]:
# 1.31 版本 baseline - 无标签重用版 (Vanilla Feature-based GCN)

import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import random
from torch_geometric.nn import GCNConv
from torch_geometric.loader import NeighborLoader
import torch_geometric.transforms as T
from ogb.nodeproppred import PygNodePropPredDataset, Evaluator

# ==========================================
# 1. 实验配置
# ==========================================
class BaseConfig:
    hidden_dim = 128
    total_hidden = 1024  
    epochs = 150
    patience = 30        
    lr_init = 0.001
    batch_size = 512
    weight_decay = 5e-4
    seeds = [1, 7, 314, 1227, 2026]

# ==========================================
# 2. 对齐型 GCN 模型
# ==========================================
class GCN_Vanilla(torch.nn.Module):
    def __init__(self, in_size, hidden_size, out_size):
        super().__init__()
        # 仅保留特征映射
        self.res_lin = nn.Linear(in_size, hidden_size)
        
        # GCN 层
        self.conv1 = GCNConv(hidden_size, hidden_size)
        self.bn1 = nn.BatchNorm1d(hidden_size)
        self.conv2 = GCNConv(hidden_size, hidden_size)
        self.bn2 = nn.BatchNorm1d(hidden_size)
        
        # 分类器
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.BatchNorm1d(hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_size // 2, out_size)
        )

    def forward(self, x, edge_index):
        # 初始特征投影
        x_proj = self.res_lin(x)
        x = x_proj
        
        # Conv 1
        x = F.relu(self.bn1(self.conv1(x, edge_index)))
        # Conv 2 + 残差链接 (特征残差)
        x = F.relu(self.bn2(self.conv2(x, edge_index))) + x_proj
        
        logits = self.classifier(x)
        return F.log_softmax(logits, dim=1)

# ==========================================
# 3. Mini-batch 评估函数 
# ==========================================
@torch.no_grad()
def inference_minibatch(model, loader, device, evaluator):
    model.eval()
    y_preds, y_trues = [], []
    for batch in loader:
        batch = batch.to(device)
        # 直接推理，不再需要 y_input
        out = model(batch.x, batch.edge_index)
        y_preds.append(out[:batch.batch_size].argmax(dim=-1).cpu())
        y_trues.append(batch.y[:batch.batch_size].cpu())
        
    y_pred = torch.cat(y_preds, dim=0).numpy().reshape(-1, 1)
    y_true = torch.cat(y_trues, dim=0).numpy().reshape(-1, 1)
    return evaluator.eval({'y_true': y_true, 'y_pred': y_pred})['acc']

# ==========================================
# 4. 实验运行器
# ==========================================
def run_experiment(seed, device):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    
    dataset = PygNodePropPredDataset(name='ogbn-arxiv', root='data', transform=T.ToUndirected())
    data = dataset[0]
    split_idx = dataset.get_idx_split()
    evaluator = Evaluator(name='ogbn-arxiv')

    train_loader = NeighborLoader(data, num_neighbors=[20, 15], batch_size=BaseConfig.batch_size,
                                 input_nodes=split_idx['train'], shuffle=True)
    valid_loader = NeighborLoader(data, num_neighbors=[-1, -1], batch_size=BaseConfig.batch_size,
                                 input_nodes=split_idx['valid'], shuffle=False)
    test_loader = NeighborLoader(data, num_neighbors=[-1, -1], batch_size=BaseConfig.batch_size,
                                input_nodes=split_idx['test'], shuffle=False)

    model = GCN_Vanilla(dataset.num_features, BaseConfig.total_hidden, dataset.num_classes).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=BaseConfig.lr_init, weight_decay=BaseConfig.weight_decay)

    best_val_acc, final_test_acc, stagnant_count = 0, 0, 0

    for epoch in range(1, BaseConfig.epochs + 1):
        model.train()
        total_loss = 0
        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            
            # 直接前向传播，不使用标签输入
            out = model(batch.x, batch.edge_index)
            loss = F.nll_loss(out[:batch.batch_size], batch.y.squeeze()[:batch.batch_size])
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        if epoch % 5 == 0:
            val_acc = inference_minibatch(model, valid_loader, device, evaluator)
            test_acc = inference_minibatch(model, test_loader, device, evaluator)
            
            if val_acc > best_val_acc:
                best_val_acc, final_test_acc, stagnant_count = val_acc, test_acc, 0
                status = "[*] NEW BEST"
            else:
                stagnant_count += 5
                status = "[ ]"
            
            print(f"Seed: {seed:4d} | Ep: {epoch:03d} | Loss: {total_loss/len(train_loader):.4f} | Val: {val_acc:.4f} | Test: {test_acc:.4f} {status}")
            if stagnant_count >= BaseConfig.patience: 
                break
            
    return final_test_acc

if __name__ == "__main__":
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    results = []
    
    print("-" * 50)
    print(f"Starting Baseline: Vanilla GCN (No Label Reuse) ({len(BaseConfig.seeds)} Seeds)")
    print("-" * 50)

    for i, s in enumerate(BaseConfig.seeds):
        print(f"\n[Run {i+1}/{len(BaseConfig.seeds)}] Seed: {s}")
        acc = run_experiment(s, device)
        results.append(acc)
    
    mean_res = np.mean(results) * 100
    std_res = np.std(results) * 100
    
    print("\n" + "=" * 50)
    print("FINAL VANILLA GCN RESULTS")
    print("-" * 50)
    print(f"Mean Accuracy: {mean_res:.2f}%")
    print(f"Std Deviation: {std_res:.2f}%")
    print(f"LaTeX Format: {mean_res:.2f} \pm {std_res:.2f}")
    print("=" * 50)

In [ ]:
# 1.31 版本 baseline - GAT  (Vanilla Feature-based GAT)

import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import random
from torch_geometric.nn import GATConv
from torch_geometric.loader import NeighborLoader
import torch_geometric.transforms as T
from ogb.nodeproppred import PygNodePropPredDataset, Evaluator

# ==========================================
# 1. 实验配置
# ==========================================
class BaseConfig:
    hidden_dim = 128     
    num_heads = 8        
    total_hidden = 1024  
    epochs = 150
    patience = 30        
    lr_init = 0.001
    batch_size = 512
    weight_decay = 5e-4
    seeds = [1, 7, 314, 1227, 2026]

# ==========================================
# 2. 对齐型 GAT 
# ==========================================
class GAT_Vanilla(torch.nn.Module):
    def __init__(self, in_size, head_dim, num_heads, out_size):
        super().__init__()
        # 初始特征投影
        self.res_lin = nn.Linear(in_size, head_dim * num_heads)
        
        # GAT 层 1: heads 拼接
        self.conv1 = GATConv(head_dim * num_heads, head_dim, heads=num_heads, dropout=0.5)
        self.bn1 = nn.BatchNorm1d(head_dim * num_heads)
        
        # GAT 层 2: heads 拼接
        self.conv2 = GATConv(head_dim * num_heads, head_dim, heads=num_heads, dropout=0.5)
        self.bn2 = nn.BatchNorm1d(head_dim * num_heads)
        
        # 分类器 (对齐 NRS 结构)
        self.classifier = nn.Sequential(
            nn.Linear(head_dim * num_heads, (head_dim * num_heads) // 2),
            nn.BatchNorm1d((head_dim * num_heads) // 2),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear((head_dim * num_heads) // 2, out_size)
        )

    def forward(self, x, edge_index):
        # 初始特征投影
        x_proj = self.res_lin(x)
        
        # Layer 1
        x = self.conv1(x_proj, edge_index)
        x = F.elu(self.bn1(x))
        
        # Layer 2 + 残差链接
        x = self.conv2(x, edge_index)
        x = F.elu(self.bn2(x)) + x_proj
        
        logits = self.classifier(x)
        return F.log_softmax(logits, dim=1)

# ==========================================
# 3. Mini-batch 评估函数
# ==========================================
@torch.no_grad()
def inference_minibatch(model, loader, device, evaluator):
    model.eval()
    y_preds, y_trues = [], []
    for batch in loader:
        batch = batch.to(device)
        out = model(batch.x, batch.edge_index)
        y_preds.append(out[:batch.batch_size].argmax(dim=-1).cpu())
        y_trues.append(batch.y[:batch.batch_size].cpu())
        
    y_pred = torch.cat(y_preds, dim=0).numpy().reshape(-1, 1)
    y_true = torch.cat(y_trues, dim=0).numpy().reshape(-1, 1)
    return evaluator.eval({'y_true': y_true, 'y_pred': y_pred})['acc']

# ==========================================
# 4. 实验运行器
# ==========================================
def run_experiment(seed, device):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    
    dataset = PygNodePropPredDataset(name='ogbn-arxiv', root='data', transform=T.ToUndirected())
    data = dataset[0]
    split_idx = dataset.get_idx_split()
    evaluator = Evaluator(name='ogbn-arxiv')

    train_loader = NeighborLoader(data, num_neighbors=[20, 15], batch_size=BaseConfig.batch_size,
                                 input_nodes=split_idx['train'], shuffle=True)
    valid_loader = NeighborLoader(data, num_neighbors=[-1, -1], batch_size=BaseConfig.batch_size,
                                 input_nodes=split_idx['valid'], shuffle=False)
    test_loader = NeighborLoader(data, num_neighbors=[-1, -1], batch_size=BaseConfig.batch_size,
                                input_nodes=split_idx['test'], shuffle=False)

    model = GAT_Vanilla(dataset.num_features, BaseConfig.hidden_dim, BaseConfig.num_heads, dataset.num_classes).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=BaseConfig.lr_init, weight_decay=BaseConfig.weight_decay)

    best_val_acc, final_test_acc, stagnant_count = 0, 0, 0

    for epoch in range(1, BaseConfig.epochs + 1):
        model.train()
        total_loss = 0
        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            out = model(batch.x, batch.edge_index)
            loss = F.nll_loss(out[:batch.batch_size], batch.y.squeeze()[:batch.batch_size])
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        if epoch % 5 == 0:
            val_acc = inference_minibatch(model, valid_loader, device, evaluator)
            test_acc = inference_minibatch(model, test_loader, device, evaluator)
            
            if val_acc > best_val_acc:
                best_val_acc, final_test_acc, stagnant_count = val_acc, test_acc, 0
                status = "[*] NEW BEST"
            else:
                stagnant_count += 5
                status = "[ ]"
            
            print(f"Seed: {seed:4d} | Ep: {epoch:03d} | Loss: {total_loss/len(train_loader):.4f} | Val: {val_acc:.4f} | Test: {test_acc:.4f} {status}")
            if stagnant_count >= BaseConfig.patience: 
                break
            
    return final_test_acc

if __name__ == "__main__":
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    results = []
    
    print("-" * 50)
    print(f"Starting Baseline: Vanilla GAT (No Label Reuse) ({len(BaseConfig.seeds)} Seeds)")
    print("-" * 50)

    for i, s in enumerate(BaseConfig.seeds):
        print(f"\n[Run {i+1}/{len(BaseConfig.seeds)}] Seed: {s}")
        acc = run_experiment(s, device)
        results.append(acc)
    
    mean_res = np.mean(results) * 100
    std_res = np.std(results) * 100
    
    print("\n" + "=" * 50)
    print("FINAL VANILLA GAT RESULTS")
    print("-" * 50)
    print(f"Mean Accuracy: {mean_res:.2f}%")
    print(f"Std Deviation: {std_res:.2f}%")
    print(f"LaTeX Format: {mean_res:.2f} \pm {std_res:.2f}")
    print("=" * 50)

/home/malina/.venv/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


--------------------------------------------------
Starting Baseline: Vanilla GAT (No Label Reuse) (5 Seeds)
--------------------------------------------------

[Run 1/5] Seed: 1


/home/malina/.venv/lib/python3.8/site-packages/ogb/nodeproppred/dataset_pyg.py:69: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.data, self.slices = torch.load(self.pro

Seed:    1 | Ep: 005 | Loss: 1.0418 | Val: 0.6609 | Test: 0.6504 [*] NEW BEST
Seed:    1 | Ep: 010 | Loss: 1.0239 | Val: 0.6804 | Test: 0.6727 [*] NEW BEST
Seed:    1 | Ep: 015 | Loss: 1.0089 | Val: 0.6752 | Test: 0.6756 [ ]
Seed:    1 | Ep: 020 | Loss: 0.9941 | Val: 0.6647 | Test: 0.6507 [ ]
Seed:    1 | Ep: 025 | Loss: 0.9779 | Val: 0.6810 | Test: 0.6750 [*] NEW BEST


KeyboardInterrupt: 

In [1]:
# 1.31 版本 baseline - Random DropEdge (Sparsity=0.3 对齐)

import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import random
from torch_geometric.nn import GATConv
from torch_geometric.loader import NeighborLoader
from torch_geometric.utils import dropout_edge
import torch_geometric.transforms as T
from ogb.nodeproppred import PygNodePropPredDataset, Evaluator

# ==========================================
# 1. 实验配置
# ==========================================
class BaseConfig:
    hidden_dim = 128     
    num_heads = 8        
    total_hidden = 1024  # 拼接后 128 * 8 = 1024
    epochs = 150
    patience = 30        
    lr_init = 0.001
    batch_size = 512
    weight_decay = 5e-4
    target_sparsity = 0.30  # 随机保留 30% 的边
    seeds = [1, 7, 314, 1227, 2026]

# ==========================================
# 2. 对齐型 GAT 模型
# ==========================================
class GAT_DropEdge(torch.nn.Module):
    def __init__(self, in_size, head_dim, num_heads, out_size):
        super().__init__()
        # 初始特征投影 (对齐 NRS 的 res_lin)
        self.res_lin = nn.Linear(in_size, head_dim * num_heads)
        
        # GAT 层 (对齐 NRS 的 GAT1 和 GAT2)
        self.conv1 = GATConv(head_dim * num_heads, head_dim, heads=num_heads, dropout=0.5)
        self.bn1 = nn.BatchNorm1d(head_dim * num_heads)
        
        self.conv2 = GATConv(head_dim * num_heads, head_dim, heads=num_heads, dropout=0.5)
        self.bn2 = nn.BatchNorm1d(head_dim * num_heads)
        
        # 分类器 (严格对齐 PANS 的 Sequential 结构)
        self.classifier = nn.Sequential(
            nn.Linear(head_dim * num_heads, (head_dim * num_heads) // 2),
            nn.BatchNorm1d((head_dim * num_heads) // 2),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear((head_dim * num_heads) // 2, out_size)
        )

    def forward(self, x, edge_index):
        x_proj = self.res_lin(x)
        
        # Layer 1
        x = self.conv1(x_proj, edge_index)
        x = F.elu(self.bn1(x))
        
        # Layer 2 + 残差链接
        x = self.conv2(x, edge_index)
        x = F.elu(self.bn2(x)) + x_proj
        
        logits = self.classifier(x)
        return F.log_softmax(logits, dim=1)

# ==========================================
# 3. Mini-batch 评估函数
# ==========================================
@torch.no_grad()
def inference_minibatch(model, loader, device, evaluator):
    model.eval()
    y_preds, y_trues = [], []
    for batch in loader:
        batch = batch.to(device)
        
        # 推理时也要保持 30% 的稀疏度
        edge_index, _ = dropout_edge(batch.edge_index, p=1 - BaseConfig.target_sparsity)
        
        out = model(batch.x, edge_index)
        y_preds.append(out[:batch.batch_size].argmax(dim=-1).cpu())
        y_trues.append(batch.y[:batch.batch_size].cpu())
        
    y_pred = torch.cat(y_preds, dim=0).numpy().reshape(-1, 1)
    y_true = torch.cat(y_trues, dim=0).numpy().reshape(-1, 1)
    return evaluator.eval({'y_true': y_true, 'y_pred': y_pred})['acc']

# ==========================================
# 4. 实验运行器
# ==========================================
def run_experiment(seed, device):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    
    dataset = PygNodePropPredDataset(name='ogbn-arxiv', root='data', transform=T.ToUndirected())
    data = dataset[0]
    split_idx = dataset.get_idx_split()
    evaluator = Evaluator(name='ogbn-arxiv')

    train_loader = NeighborLoader(data, num_neighbors=[20, 15], batch_size=BaseConfig.batch_size,
                                 input_nodes=split_idx['train'], shuffle=True)
    valid_loader = NeighborLoader(data, num_neighbors=[-1, -1], batch_size=BaseConfig.batch_size,
                                 input_nodes=split_idx['valid'], shuffle=False)
    test_loader = NeighborLoader(data, num_neighbors=[-1, -1], batch_size=BaseConfig.batch_size,
                                input_nodes=split_idx['test'], shuffle=False)

    model = GAT_DropEdge(dataset.num_features, BaseConfig.hidden_dim, BaseConfig.num_heads, dataset.num_classes).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=BaseConfig.lr_init, weight_decay=BaseConfig.weight_decay)

    best_val_acc, final_test_acc, stagnant_count = 0, 0, 0

    for epoch in range(1, BaseConfig.epochs + 1):
        model.train()
        total_loss = 0
        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            
            # 每一轮训练都随机删除 70% 的边
            edge_index, _ = dropout_edge(batch.edge_index, p=1 - BaseConfig.target_sparsity)
            
            out = model(batch.x, edge_index)
            loss = F.nll_loss(out[:batch.batch_size], batch.y.squeeze()[:batch.batch_size])
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        if epoch % 5 == 0:
            val_acc = inference_minibatch(model, valid_loader, device, evaluator)
            test_acc = inference_minibatch(model, test_loader, device, evaluator)
            
            if val_acc > best_val_acc:
                best_val_acc, final_test_acc, stagnant_count = val_acc, test_acc, 0
                status = "[*] NEW BEST"
            else:
                stagnant_count += 5
                status = "[ ]"
            
            print(f"Seed: {seed:4d} | Ep: {epoch:03d} | Loss: {total_loss/len(train_loader):.4f} | Val: {val_acc:.4f} | Test: {test_acc:.4f} {status}")
            if stagnant_count >= BaseConfig.patience: break
            
    return final_test_acc

if __name__ == "__main__":
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    results = []
    print(f"Starting Baseline: GAT + Random DropEdge (Target Sparsity: {BaseConfig.target_sparsity})")
    for s in BaseConfig.seeds:
        acc = run_experiment(s, device)
        results.append(acc)
    
    print("\n" + "="*40)
    print(f"Final GAT-DropEdge Results: {np.mean(results)*100:.2f} ± {np.std(results)*100:.2f}")
    print("="*40)

/home/malina/.venv/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Starting Baseline: GAT + Random DropEdge (Target Sparsity: 0.3)


/home/malina/.venv/lib/python3.8/site-packages/ogb/nodeproppred/dataset_pyg.py:69: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.data, self.slices = torch.load(self.pro

Seed:    1 | Ep: 005 | Loss: 1.3382 | Val: 0.6311 | Test: 0.6227 [*] NEW BEST
Seed:    1 | Ep: 010 | Loss: 1.3199 | Val: 0.6225 | Test: 0.6219 [ ]
Seed:    1 | Ep: 015 | Loss: 1.3001 | Val: 0.6352 | Test: 0.6380 [*] NEW BEST
Seed:    1 | Ep: 020 | Loss: 1.2872 | Val: 0.6313 | Test: 0.6315 [ ]


KeyboardInterrupt: 

In [1]:
# 2.1 ogbn-arxiv baseline top-k pruning


import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import random
from torch_geometric.nn import GATConv
from torch_geometric.loader import NeighborLoader
import torch_geometric.transforms as T
from ogb.nodeproppred import PygNodePropPredDataset, Evaluator

# ==========================================
# 1. 实验配置
# ==========================================
class BaseConfig:
    hidden_dim = 128     
    num_heads = 8        
    total_hidden = 1024  
    epochs = 150
    patience = 30        
    lr_init = 0.001
    batch_size = 512
    weight_decay = 5e-4
    target_sparsity = 0.30  
    seeds = [1, 7, 314, 1227, 2026]

# ==========================================
# 2. 剪枝模型
# ==========================================
class GAT_Pruning(torch.nn.Module):
    def __init__(self, in_size, head_dim, num_heads, out_size):
        super().__init__()
        self.res_lin = nn.Linear(in_size, head_dim * num_heads)
        
        # 对齐 NRS 的 GAT 配置
        self.conv1 = GATConv(head_dim * num_heads, head_dim, heads=num_heads, dropout=0.5)
        self.bn1 = nn.BatchNorm1d(head_dim * num_heads)
        
        self.conv2 = GATConv(head_dim * num_heads, head_dim, heads=num_heads, dropout=0.5)
        self.bn2 = nn.BatchNorm1d(head_dim * num_heads)
        
        self.classifier = nn.Sequential(
            nn.Linear(head_dim * num_heads, (head_dim * num_heads) // 2),
            nn.BatchNorm1d((head_dim * num_heads) // 2),
            nn.ReLU(), nn.Dropout(0.3),
            nn.Linear((head_dim * num_heads) // 2, out_size)
        )

    def prune_edges(self, edge_index, edge_attr, sparsity):
        """
        基于attention权重进行 top-k 剪枝
        """
        if edge_attr is None or edge_index.numel() == 0:
            return edge_index

        score = edge_attr.mean(dim=-1)
        num_edges = edge_index.size(1)
        
        # 2. 确保 1 <= k <= num_edges，防止 k=0 崩溃
        k = int(num_edges * sparsity)
        k = max(1, min(k, num_edges))
        
        # 3. 获取索引
        _, top_indices = torch.topk(score, k, sorted=False)
        return edge_index[:, top_indices]

    def forward(self, x, edge_index):
        x_proj = self.res_lin(x)
        

        x, (edge_index_tmp, alpha) = self.conv1(x_proj, edge_index, return_attention_weights=True)
        x = F.elu(self.bn1(x))
        
        pruned_edge_index = self.prune_edges(edge_index_tmp, alpha, BaseConfig.target_sparsity)
        
        # 第二层聚合
        x = self.conv2(x, pruned_edge_index)
        x = F.elu(self.bn2(x)) + x_proj
        
        logits = self.classifier(x)
        return F.log_softmax(logits, dim=1)

# ==========================================
# 3. Mini-batch 评估函数
# ==========================================
@torch.no_grad()
def inference_minibatch(model, loader, device, evaluator):
    model.eval()
    y_preds, y_trues = [], []
    for batch in loader:
        batch = batch.to(device)
        out = model(batch.x, batch.edge_index)
        y_preds.append(out[:batch.batch_size].argmax(dim=-1).cpu())
        y_trues.append(batch.y[:batch.batch_size].cpu())
        
    y_pred = torch.cat(y_preds, dim=0).numpy().reshape(-1, 1)
    y_true = torch.cat(y_trues, dim=0).numpy().reshape(-1, 1)
    return evaluator.eval({'y_true': y_true, 'y_pred': y_pred})['acc']

# ==========================================
# 4. 实验运行
# ==========================================
def run_experiment(seed, device):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    
    dataset = PygNodePropPredDataset(name='ogbn-arxiv', root='data', transform=T.ToUndirected())
    data = dataset[0]
    split_idx = dataset.get_idx_split()
    evaluator = Evaluator(name='ogbn-arxiv')

    train_loader = NeighborLoader(data, num_neighbors=[20, 15], batch_size=BaseConfig.batch_size,
                                 input_nodes=split_idx['train'], shuffle=True)
    valid_loader = NeighborLoader(data, num_neighbors=[-1, -1], batch_size=BaseConfig.batch_size,
                                 input_nodes=split_idx['valid'], shuffle=False)
    test_loader = NeighborLoader(data, num_neighbors=[-1, -1], batch_size=BaseConfig.batch_size,
                                input_nodes=split_idx['test'], shuffle=False)

    model = GAT_Pruning(dataset.num_features, BaseConfig.hidden_dim, BaseConfig.num_heads, dataset.num_classes).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=BaseConfig.lr_init, weight_decay=BaseConfig.weight_decay)

    best_val_acc, final_test_acc, stagnant_count = 0, 0, 0

    for epoch in range(1, BaseConfig.epochs + 1):
        model.train()
        total_loss = 0
        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            out = model(batch.x, batch.edge_index)
            loss = F.nll_loss(out[:batch.batch_size], batch.y.squeeze()[:batch.batch_size])
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        if epoch % 5 == 0:
            val_acc = inference_minibatch(model, valid_loader, device, evaluator)
            test_acc = inference_minibatch(model, test_loader, device, evaluator)
            if val_acc > best_val_acc:
                best_val_acc, final_test_acc, stagnant_count = val_acc, test_acc, 0
                status = "[*] NEW BEST"
            else:
                stagnant_count += 5
                status = "[ ]"
            print(f"Seed: {seed:4d} | Ep: {epoch:03d} | Loss: {total_loss/len(train_loader):.4f} | Val: {val_acc:.4f} | Test: {test_acc:.4f} {status}")
            if stagnant_count >= BaseConfig.patience: break
            
    return final_test_acc

if __name__ == "__main__":
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    results = []
    print(f"Starting Baseline: GAT + Top-K Pruning (Target Sparsity: {BaseConfig.target_sparsity})")
    for s in BaseConfig.seeds:
        acc = run_experiment(s, device)
        results.append(acc)
    
    print("\n" + "="*40)
    print(f"Final GAT-Pruning Results: {np.mean(results)*100:.2f} ± {np.std(results)*100:.2f}")
    print("="*40) 

/home/malina/.venv/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Starting Baseline: GAT + Top-K Pruning (Target Sparsity: 0.3)


/home/malina/.venv/lib/python3.8/site-packages/ogb/nodeproppred/dataset_pyg.py:69: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.data, self.slices = torch.load(self.pro

Seed:    1 | Ep: 005 | Loss: 1.1729 | Val: 0.6880 | Test: 0.6833 [*] NEW BEST
Seed:    1 | Ep: 010 | Loss: 1.1544 | Val: 0.6635 | Test: 0.6430 [ ]
Seed:    1 | Ep: 015 | Loss: 1.1392 | Val: 0.6539 | Test: 0.6534 [ ]
Seed:    1 | Ep: 020 | Loss: 1.1230 | Val: 0.6479 | Test: 0.6195 [ ]
Seed:    1 | Ep: 025 | Loss: 1.1097 | Val: 0.6670 | Test: 0.6456 [ ]
Seed:    1 | Ep: 030 | Loss: 1.0980 | Val: 0.6785 | Test: 0.6770 [ ]
Seed:    1 | Ep: 035 | Loss: 1.0894 | Val: 0.6902 | Test: 0.6845 [*] NEW BEST
Seed:    1 | Ep: 040 | Loss: 1.0823 | Val: 0.6940 | Test: 0.6840 [*] NEW BEST
Seed:    1 | Ep: 045 | Loss: 1.0769 | Val: 0.6864 | Test: 0.6742 [ ]
Seed:    1 | Ep: 050 | Loss: 1.0755 | Val: 0.6768 | Test: 0.6556 [ ]


KeyboardInterrupt: 

In [1]:
# 1.31更新 ogbn baseline NeuralSparse 


import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import random
from torch_geometric.nn import GATConv
from torch_geometric.loader import NeighborLoader
import torch_geometric.transforms as T
from ogb.nodeproppred import PygNodePropPredDataset, Evaluator

# ==========================================
# 1. 实验配置
# ==========================================
class BaseConfig:
    hidden_dim = 128     
    num_heads = 8        
    total_hidden = 1024  
    epochs = 150
    patience = 30        
    lr_init = 0.001
    batch_size = 512
    target_sparsity = 0.30  
    weight_decay = 5e-4
    seeds = [1, 7, 314, 1227, 2026]

# ==========================================
# 2. NeuralSparse 
# ==========================================
class NeuralSparse_Sampler(nn.Module):
    def __init__(self, h_dim):
        super().__init__()
        # 学习边的评分逻辑
        self.scorer = nn.Sequential(
            nn.Linear(h_dim * 2, h_dim // 2),
            nn.ReLU(),
            nn.Linear(h_dim // 2, 1)
        )

    def forward(self, h, edge_index, tau=1.0, hard=True):
        row, col = edge_index
        edge_feat = torch.cat([h[row], h[col]], dim=-1)
        logits = self.scorer(edge_feat).squeeze()
        
        # 二分类采样空间：[Discard, Keep]
        sampling_logits = torch.stack([torch.zeros_like(logits), logits], dim=-1)
        
        # Gumbel-Softmax 重参数化采样
        weights = F.gumbel_softmax(sampling_logits, tau=tau, hard=hard, dim=-1)[:, 1]
        return weights, logits

class NeuralSparse_System(torch.nn.Module):
    def __init__(self, in_size, hidden_size, out_size):
        super().__init__()
        self.total_hidden = hidden_size * 8
        self.res_lin = nn.Linear(in_size, self.total_hidden)
        
        self.gat1 = GATConv(self.total_hidden, hidden_size, heads=8, dropout=0.5)
        self.bn1 = nn.BatchNorm1d(self.total_hidden)
        self.gat2 = GATConv(self.total_hidden, hidden_size, heads=8, dropout=0.5)
        self.bn2 = nn.BatchNorm1d(self.total_hidden)
        
        self.sampler_net = NeuralSparse_Sampler(self.total_hidden)
        
        self.classifier = nn.Sequential(
            nn.Linear(self.total_hidden, hidden_size * 4),
            nn.BatchNorm1d(hidden_size * 4),
            nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(hidden_size * 4, out_size)
        )

    def forward(self, x, edge_index, tau=1.0, hard=True):
        x_proj = self.res_lin(x)
        
        # 1. 特征编码
        h1 = F.elu(self.bn1(self.gat1(x_proj, edge_index)))
        h_base = F.elu(self.bn2(self.gat2(h1, edge_index)))
        
        # 2. NeuralSparse 采样 (单步，非递归)
        weights, logits_raw = self.sampler_net(h_base, edge_index, tau=tau, hard=hard)
        
        # 3. 稀疏聚合
        row, col = edge_index
        h_sparse = h_base + (weights.view(-1, 1) * h_base[col]).new_zeros(h_base.shape).index_add_(0, row, weights.view(-1, 1) * h_base[col])
        
        logits = self.classifier(h_sparse)
        return F.log_softmax(logits, dim=1), weights, logits_raw

# ==========================================
# 3. 训练与评估逻辑
# ==========================================
def train_step(model, loader, optimizer, device, epoch):
    model.train()
    total_loss = 0
    # Tau 退火：模仿 PANS 的训练过程
    tau = max(0.2, 1.0 - (epoch / 50)) 
    
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        
        log_probs, weights, _ = model(batch.x, batch.edge_index, tau=tau, hard=(epoch > 50))
        
        # 分类损失
        loss_clf = F.nll_loss(log_probs[:batch.batch_size], batch.y.squeeze()[:batch.batch_size])
        
        # 稀疏度约束：拉近平均权重与 target_sparsity
        loss_sp = F.l1_loss(weights.mean(), torch.tensor(BaseConfig.target_sparsity, device=device))
        
        loss = loss_clf + 1.0 * loss_sp # 1.0 为 Sparsity Penalty 权重
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

@torch.no_grad()
def evaluate(model, loader, device, evaluator):
    model.eval()
    y_preds, y_trues = [], []
    for batch in loader:
        batch = batch.to(device)
        lp, _, _ = model(batch.x, batch.edge_index, hard=True)
        y_preds.append(lp[:batch.batch_size].argmax(dim=-1).cpu())
        y_trues.append(batch.y[:batch.batch_size].cpu())
        
    y_pred = torch.cat(y_preds, dim=0).numpy().reshape(-1, 1)
    y_true = torch.cat(y_trues, dim=0).numpy().reshape(-1, 1)
    return evaluator.eval({'y_true': y_true, 'y_pred': y_pred})['acc']

# ==========================================
# 4. 实验主循环
# ==========================================
def run_experiment(seed, device):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    
    dataset = PygNodePropPredDataset(name='ogbn-arxiv', root='data', transform=T.ToUndirected())
    data = dataset[0]
    split_idx = dataset.get_idx_split()
    evaluator = Evaluator(name='ogbn-arxiv')

    train_loader = NeighborLoader(data, num_neighbors=[20, 15], batch_size=BaseConfig.batch_size,
                                 input_nodes=split_idx['train'], shuffle=True)
    valid_loader = NeighborLoader(data, num_neighbors=[-1, -1], batch_size=BaseConfig.batch_size,
                                 input_nodes=split_idx['valid'], shuffle=False)
    test_loader = NeighborLoader(data, num_neighbors=[-1, -1], batch_size=BaseConfig.batch_size,
                                input_nodes=split_idx['test'], shuffle=False)

    model = NeuralSparse_System(dataset.num_features, BaseConfig.hidden_dim, dataset.num_classes).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=BaseConfig.lr_init, weight_decay=BaseConfig.weight_decay)

    best_val, final_test, stagnant = 0, 0, 0
    for epoch in range(1, BaseConfig.epochs + 1):
        loss = train_step(model, train_loader, optimizer, device, epoch)
        
        if epoch % 5 == 0:
            val_acc = evaluate(model, valid_loader, device, evaluator)
            test_acc = evaluate(model, test_loader, device, evaluator)
            
            if val_acc > best_val:
                best_val, final_test, stagnant = val_acc, test_acc, 0
                status = "[*] NEW BEST"
            else:
                stagnant += 5; status = "[ ]"
            
            print(f"Seed: {seed} | Ep: {epoch:03d} | Val: {val_acc:.4f} | Test: {test_acc:.4f} {status}")
            if stagnant >= BaseConfig.patience: break
            
    return final_test

if __name__ == "__main__":
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    results = []
    print(f"Starting Baseline: NeuralSparse (Target Sparsity: {BaseConfig.target_sparsity})")
    for s in BaseConfig.seeds:
        acc = run_experiment(s, device)
        results.append(acc)
    
    print(f"\nFinal NeuralSparse Results: {np.mean(results)*100:.2f} ± {np.std(results)*100:.2f}")

/home/malina/.venv/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Starting Baseline: NeuralSparse (Target Sparsity: 0.3)


/home/malina/.venv/lib/python3.8/site-packages/ogb/nodeproppred/dataset_pyg.py:69: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.data, self.slices = torch.load(self.pro

Seed: 1 | Ep: 005 | Val: 0.6505 | Test: 0.6366 [*] NEW BEST
Seed: 1 | Ep: 010 | Val: 0.6212 | Test: 0.5908 [ ]
Seed: 1 | Ep: 015 | Val: 0.6794 | Test: 0.6767 [*] NEW BEST
Seed: 1 | Ep: 020 | Val: 0.6547 | Test: 0.6525 [ ]
Seed: 1 | Ep: 025 | Val: 0.6824 | Test: 0.6705 [*] NEW BEST
Seed: 1 | Ep: 030 | Val: 0.6746 | Test: 0.6693 [ ]
Seed: 1 | Ep: 035 | Val: 0.6560 | Test: 0.6319 [ ]


KeyboardInterrupt: 

In [1]:
# GSAT 非递归 baseline

# ogbn-arxiv Baseline: Graph Stochastic Attention (GSAT)
import os
import os.path as osp
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv, MessagePassing
from torch_geometric.utils import negative_sampling, softmax, add_self_loops
from torch_geometric.loader import NeighborLoader 
import torch_geometric.transforms as T
from ogb.nodeproppred import PygNodePropPredDataset, Evaluator 
import numpy as np
import matplotlib.pyplot as plt

# ==========================================
# 1. Global 参数配置 (新增 GSAT 专属参数)
# ==========================================
class Config:
    hidden_dim = 128     
    proj_dim = 256       
    epochs = 150         
    soft_end = 50        
    lr_init = 0.001      
    batch_size = 512    
    weight_decay = 5e-4    
    patience = 7            
    # [GSAT MODIFIED] 论文核心超参
    beta = 0.3            # 信息瓶颈惩罚系数 (Eq. 8) [cite: 212]
    r = 0.3               # 边保留先验概率 (Eq. 9) [cite: 227, 337]

# ==========================================
# 2. GSAT 核心架构组件
# ==========================================

class GSAT_Sampler(nn.Module):
    """ [GSAT MODIFIED] 实现了论文 Sec 4.2 的随机注意力机制 """
    def __init__(self, in_dim, proj_dim=128):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(in_dim * 2, proj_dim),
            nn.ReLU(),
            nn.Linear(proj_dim, 1)
        )
        
    def forward(self, h_i, h_j, tau=1.0, training=True):
        edge_feat = torch.cat([h_i, h_j], dim=-1)
        logits = self.mlp(edge_feat).squeeze(-1) # p_uv 的未归一化对数
        
        # 构造 Bernoulli 采样的对数概率: [Discard, Keep]
        # [cite: 219, 224]
        sampling_logits = torch.stack([torch.zeros_like(logits), logits], dim=-1)
        
        if training:
            # Gumbel-Softmax 重参数化采样 (Bernoulli 近似) 
            weights = F.gumbel_softmax(sampling_logits, tau=tau, hard=True, dim=-1)[:, 1]
        else:
            # 测试阶段根据学习到的 p_uv 直接取结果 [cite: 241]
            weights = (torch.sigmoid(logits) > 0.5).float()
            
        return weights, logits

class GSATSystem(torch.nn.Module):
    def __init__(self, in_size, hidden_size, out_size):
        super().__init__()
        self.total_hidden = hidden_size * 8
        self.res_lin = nn.Linear(in_size, self.total_hidden)
        
        # Extractor g_phi 和 Predictor f_theta 共享 GNN 参数 [cite: 53, 216]
        self.gnn_shared = nn.ModuleList([
            GATConv(self.total_hidden, hidden_size, heads=8, dropout=0.5),
            GATConv(self.total_hidden, hidden_size, heads=8, dropout=0.5)
        ])
        self.bns = nn.ModuleList([nn.BatchNorm1d(self.total_hidden) for _ in range(2)])
        
        self.sampler_net = GSAT_Sampler(in_dim=self.total_hidden)
        
        self.classifier = nn.Sequential(
            nn.Linear(self.total_hidden, hidden_size * 4),
            nn.BatchNorm1d(hidden_size * 4),
            nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(hidden_size * 4, out_size)
        )

    def forward(self, x, edge_index, tau=1.0):
        h = self.res_lin(x)
        
        # 1. 语义发现: 编码输入图 G [cite: 50, 217]
        for i, conv in enumerate(self.gnn_shared):
            h = F.elu(self.bns[i](conv(h, edge_index)))
        
        # 2. GSAT 随机采样: 注入随机性以阻塞任务无关信息 [cite: 10, 62]
        # 注意：GSAT 通常为单步执行 [cite: 51]
        weights, logits_raw = self.sampler_net(h[edge_index[0]], h[edge_index[1]], tau=tau, training=self.training)
        
        # 3. 预测阶段: 在扰动图 G_S 上进行计算 [cite: 50, 51]
        row, col = edge_index
        # 稀疏聚合逻辑 (GSAT 不显式增强特征，仅过滤边信息流)
        # 这里模拟 GIB 后的信息约束聚合
        h_s = torch.zeros_like(h).index_add_(0, row, weights.view(-1, 1) * h[col])
        
        logits = self.classifier(h_s + h) # 残差结合
        return F.log_softmax(logits, dim=1), weights, logits_raw

# ==========================================
# 3. 核心训练函数 (引入 KL Regularization)
# ==========================================

def get_kl_loss(p_uv_logits, r):
    """ [GSAT MODIFIED] 计算论文 Eq. 9 描述的 KL 散度项  """
    p = torch.sigmoid(p_uv_logits)
    # 计算 KL(Bern(p) || Bern(r))
    kl = p * torch.log(p / r + 1e-9) + (1 - p) * torch.log((1 - p) / (1 - r) + 1e-9)
    return kl.mean()

def train_minibatch(epoch):
    model.train()
    total_loss_epoch, total_clf_loss, total_kl_loss = 0, 0, 0
    
    # Tau 退火 (对齐 Gumbel-Softmax)
    tau = max(0.2, 1.0 - (epoch / 50)) 

    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()

        log_probs, weights, logits_raw = model(batch.x, batch.edge_index, tau=tau)
        
        # 1. 分类损失 [cite: 232]
        loss_clf = F.nll_loss(log_probs[:batch.batch_size], batch.y.squeeze()[:batch.batch_size])
        
        # 2. 信息瓶颈 KL 约束 (GSAT 核心惩罚) [cite: 212, 239]
        loss_kl = get_kl_loss(logits_raw, Config.r)
        
        # 3. 最终 GSAT 目标函数 [cite: 212]
        total_loss = loss_clf + Config.beta * loss_kl
        
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        total_loss_epoch += total_loss.item()
        total_clf_loss += loss_clf.item()
        total_kl_loss += loss_kl.item()

    return total_loss_epoch/len(train_loader), total_clf_loss/len(train_loader), total_kl_loss/len(train_loader)

# [数据加载逻辑与原脚本一致，仅将 model 指向 GSATSystem]
@torch.no_grad()
def inference_minibatch(loader):
    model.eval() # [cite: 240, 241] 切换到评估模式，GSAT 会自动关闭随机性
    y_preds, y_trues = [], []
    
    for batch in loader:
        batch = batch.to(device)
        # 执行前向传播，此时模型内部 sampler 会执行 (sigmoid(logits) > 0.5) 
        log_probs, _, _ = model(batch.x, batch.edge_index)
        
        # 只保留中心节点的预测结果 [cite: 225]
        y_preds.append(log_probs[:batch.batch_size].argmax(dim=-1).cpu())
        y_trues.append(batch.y[:batch.batch_size].cpu())
        
    y_pred = torch.cat(y_preds, dim=0).numpy().reshape(-1, 1)
    y_true = torch.cat(y_trues, dim=0).numpy().reshape(-1, 1)
    
    return evaluator.eval({'y_true': y_true, 'y_pred': y_pred})['acc']
# ==========================================
# 4. 数据加载与主循环 (GSAT 版)
# ==========================================

import torch.serialization
torch.serialization.add_safe_globals([set, dict, list, tuple])

dataset = PygNodePropPredDataset(name='ogbn-arxiv', root='data', transform=T.ToUndirected())
data = dataset[0]

# [MODIFIED] GSAT 通常不需要显式添加自环，因为其学习的是边重要性概率 p_uv
# 但为了与你的原始流程对齐并保持 GNN 稳定性，我们保留此步骤
edge_index, _ = add_self_loops(data.edge_index, num_nodes=data.num_nodes)
data.edge_index = edge_index

split_idx = dataset.get_idx_split()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 训练用 Loader: 保持采样深度对齐 [20, 15]
train_loader = NeighborLoader(data, num_neighbors=[20, 15], batch_size=Config.batch_size, 
                              input_nodes=split_idx['train'], shuffle=True)

# 推理用 Loader: 看全邻居以保证评估准确性
valid_loader = NeighborLoader(data, num_neighbors=[-1, -1], batch_size=Config.batch_size, 
                              input_nodes=split_idx['valid'], shuffle=False)
test_loader = NeighborLoader(data, num_neighbors=[-1, -1], batch_size=Config.batch_size, 
                             input_nodes=split_idx['test'], shuffle=False)

# 初始化 GSAT 系统
model = GSATSystem(dataset.num_features, Config.hidden_dim, dataset.num_classes).to(device)

# [GSAT MODIFIED] 差分学习率应用
# 采样器学习率通常比主网络低 (0.1x)，以保证信息瓶颈收敛的稳定性 [cite: 645, 651]
sampler_params = list(model.sampler_net.parameters())
other_params = [p for n, p in model.named_parameters() if 'sampler_net' not in n]

optimizer = torch.optim.Adam([
    {'params': other_params, 'lr': Config.lr_init}, 
    {'params': sampler_params, 'lr': Config.lr_init * 0.1}
], weight_decay=Config.weight_decay)

evaluator = Evaluator(name='ogbn-arxiv')
scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[30, 60], gamma=0.5)

# 历史统计容器，新增 kl_loss 以监控信息瓶颈
history = {'train_loss': [], 'clf_loss': [], 'kl_loss': [], 'val_acc': [], 'test_acc': [], 'sparsity': []}
best_valid_acc, stagnant_epochs = 0.0, 0

print(f"Starting GSAT Baseline: Stochastic Attention + IB Regularization (Beta: {Config.beta})...")

for epoch in range(1, Config.epochs + 1):
    # 调用之前定义的 GSAT 专属训练函数
    avg_loss, avg_clf, avg_kl = train_minibatch(epoch)
    scheduler.step()
    
    # 记录损失与当前边保留率 (Sparsity)
    history['train_loss'].append(avg_loss)
    history['clf_loss'].append(avg_clf)
    history['kl_loss'].append(avg_kl)
    
    # 获取当前 Epoch 的平均边保留概率作为 Sparsity 参考
    # 这一步仅作日志记录，不参与计算
    model.eval()
    with torch.no_grad():
        # 这里简单抽取一个 Batch 计算 Sparsity 展示
        sample_batch = next(iter(train_loader)).to(device)
        _, weights, _ = model(sample_batch.x, sample_batch.edge_index)
        avg_sp = weights.mean().item()
        history['sparsity'].append(avg_sp)

    if epoch % 5 == 0: 
        # 调用推理函数 (GSAT 模式下推理会自动关闭 Stochasticity) 
        valid_acc = inference_minibatch(valid_loader)
        test_acc = inference_minibatch(test_loader)
        history['val_acc'].append(valid_acc)
        history['test_acc'].append(test_acc)
        
        if valid_acc > best_valid_acc:
            best_valid_acc = valid_acc
            stagnant_epochs = 0
            torch.save(model.state_dict(), 'best_gsat_arxiv.pt')
            status = "[*] NEW BEST"
        else:
            stagnant_epochs += 1
            status = "[ ]"
            
        print(f"\n{status} Ep: {epoch:03d} | Val:{valid_acc:.4f} | Test:{test_acc:.4f} | KL_L:{avg_kl:.4f} | SP:{avg_sp:.4f}")
        
        if stagnant_epochs >= Config.patience: 
            print(f"\n[Early Stop] Triggered at epoch {epoch}")
            break
        torch.cuda.empty_cache()
    else:
        # 实时打印训练进度
        print(f"      Ep: {epoch:03d} | Clf_L: {avg_clf:.4f} | KL_L: {avg_kl:.4f} | SP: {avg_sp:.4f}", end='\r')



/home/malina/.venv/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/malina/.venv/lib/python3.8/site-packages/ogb/nodeproppred/dataset_pyg.py:69: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. 

Starting GSAT Baseline: Stochastic Attention + IB Regularization (Beta: 0.3)...
      Ep: 004 | Clf_L: 1.0763 | KL_L: 0.0253 | SP: 0.0933
[*] NEW BEST Ep: 005 | Val:0.6429 | Test:0.6173 | KL_L:0.0262 | SP:0.2670
      Ep: 009 | Clf_L: 1.0540 | KL_L: 0.0289 | SP: 0.1826
[ ] Ep: 010 | Val:0.6057 | Test:0.5648 | KL_L:0.0295 | SP:0.4088
      Ep: 014 | Clf_L: 1.0319 | KL_L: 0.0342 | SP: 0.4926
[*] NEW BEST Ep: 015 | Val:0.6562 | Test:0.6459 | KL_L:0.0357 | SP:0.2297
      Ep: 019 | Clf_L: 1.0135 | KL_L: 0.0392 | SP: 0.5144
[*] NEW BEST Ep: 020 | Val:0.6837 | Test:0.6676 | KL_L:0.0404 | SP:0.8182
      Ep: 024 | Clf_L: 0.9966 | KL_L: 0.0431 | SP: 0.8761
[ ] Ep: 025 | Val:0.6336 | Test:0.6164 | KL_L:0.0449 | SP:0.9428
      Ep: 029 | Clf_L: 0.9830 | KL_L: 0.0479 | SP: 0.7130
[ ] Ep: 030 | Val:0.6779 | Test:0.6559 | KL_L:0.0503 | SP:0.8911
      Ep: 034 | Clf_L: 0.9404 | KL_L: 0.0554 | SP: 0.9645
[*] NEW BEST Ep: 035 | Val:0.6853 | Test:0.6715 | KL_L:0.0555 | SP:0.9477
      Ep: 039 | Clf_L: 

KeyboardInterrupt: 

In [ ]:
# ogbn-arxiv Baseline: Graph Stochastic Attention (GSAT) 
# 实施建议：Beta Warmup + IB Regularization + Sparsity Alignment

import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv, MessagePassing
from torch_geometric.utils import negative_sampling, softmax, add_self_loops
from torch_geometric.loader import NeighborLoader 
import torch_geometric.transforms as T
from ogb.nodeproppred import PygNodePropPredDataset, Evaluator 
import numpy as np

# ==========================================
# 1. Global 参数配置 (GSAT 论文超参对齐)
# ==========================================
class Config:
    hidden_dim = 128     
    proj_dim = 256       
    epochs = 150         
    lr_init = 0.001      
    batch_size = 512    
    weight_decay = 5e-4    
    patience = 15           # 稍微调大耐心值，因为 GSAT 在 warmup 后才开始收敛
    
    # [GSAT核心对齐] [cite: 212, 227]
    beta = 0.5             # 信息瓶颈强度 (建议 0.1 - 1.0 之间微调)
    r = 0.3                # 先验保留率 (对齐你的稀疏度 0.3)
    warmup_epochs = 10     # 前 10 轮不进行信息压缩

# ==========================================
# 2. GSAT 核心架构
# ==========================================

class GSAT_Sampler(nn.Module):
    """ 实现论文 Sec 4.2 的随机注意力机制 [cite: 217, 218] """
    def __init__(self, in_dim, proj_dim=128):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(in_dim * 2, proj_dim),
            nn.ReLU(),
            nn.Linear(proj_dim, 1)
        )
        
    def forward(self, h_i, h_j, tau=1.0, training=True):
        edge_feat = torch.cat([h_i, h_j], dim=-1)
        logits = self.mlp(edge_feat).squeeze(-1)
        
        # 构造 Bernoulli 采样概率 [cite: 219, 224]
        sampling_logits = torch.stack([torch.zeros_like(logits), logits], dim=-1)
        
        if training:
            # 注入随机性：Gumbel-Softmax 离散采样 [cite: 62, 220]
            weights = F.gumbel_softmax(sampling_logits, tau=tau, hard=True, dim=-1)[:, 1]
        else:
            # 测试阶段：取确定性 Top-k 边 (KDD严谨对比建议) [cite: 241]
            # 为了保证绝对公平，可以在此处强制保留得分前 30% 的边
            probs = torch.sigmoid(logits)
            weights = (probs > 0.5).float() # 或者使用 torch.topk
            
        return weights, logits

class GSATSystem(torch.nn.Module):
    def __init__(self, in_size, hidden_size, out_size):
        super().__init__()
        self.total_hidden = hidden_size * 8
        self.res_lin = nn.Linear(in_size, self.total_hidden)
        
        # Extractor 与 Predictor 共享 GNN 骨干 [cite: 53, 216]
        self.gnn = nn.ModuleList([
            GATConv(self.total_hidden, hidden_size, heads=8, dropout=0.5),
            GATConv(self.total_hidden, hidden_size, heads=8, dropout=0.5)
        ])
        self.bns = nn.ModuleList([nn.BatchNorm1d(self.total_hidden) for _ in range(2)])
        self.sampler_net = GSAT_Sampler(in_dim=self.total_hidden)
        
        self.classifier = nn.Sequential(
            nn.Linear(self.total_hidden, hidden_size * 4),
            nn.BatchNorm1d(hidden_size * 4),
            nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(hidden_size * 4, out_size)
        )

    def forward(self, x, edge_index, tau=1.0):
        h = self.res_lin(x)
        # 编码原始图信息
        for i, conv in enumerate(self.gnn):
            h = F.elu(self.bns[i](conv(h, edge_index)))
        
        # GSAT 采样：基于 IB 原理过滤边信息 [cite: 10, 239]
        weights, logits_raw = self.sampler_net(h[edge_index[0]], h[edge_index[1]], 
                                              tau=tau, training=self.training)
        
        # 在扰动后的子图 G_S 上聚合信息 [cite: 50]
        row, col = edge_index
        h_s = torch.zeros_like(h).index_add_(0, row, weights.view(-1, 1) * h[col])
        
        logits = self.classifier(h_s + h)
        return F.log_softmax(logits, dim=1), weights, logits_raw

# ==========================================
# 3. 损失函数与训练逻辑
# ==========================================

def get_kl_loss(p_uv_logits, r):
    """ 计算信息瓶颈正则项 (Eq. 9) [cite: 236] """
    p = torch.sigmoid(p_uv_logits)
    # KL(Bern(p) || Bern(r))
    kl = p * torch.log(p / r + 1e-9) + (1 - p) * torch.log((1 - p) / (1 - r) + 1e-9)
    return kl.mean()

def train_minibatch(epoch):
    model.train()
    total_loss_epoch, total_clf_loss, total_kl_loss = 0, 0, 0
    
    # [新增] 参数预热逻辑：前 Config.warmup_epochs 轮 beta 为 0
    curr_beta = 0 if epoch <= Config.warmup_epochs else \
                min(Config.beta, Config.beta * (epoch - Config.warmup_epochs) / 50)
    
    # Gumbel-Softmax 温度退火
    tau = max(0.1, 1.0 - (epoch / 50)) 

    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        
        log_probs, weights, logits_raw = model(batch.x, batch.edge_index, tau=tau)
        
        loss_clf = F.nll_loss(log_probs[:batch.batch_size], batch.y.squeeze()[:batch.batch_size])
        loss_kl = get_kl_loss(logits_raw, Config.r) # [cite: 233, 239]
        
        # IB 目标函数 
        total_loss = loss_clf + curr_beta * loss_kl
        
        total_loss.backward()
        optimizer.step()
        
        total_loss_epoch += total_loss.item()
        total_clf_loss += loss_clf.item()
        total_kl_loss += loss_kl.item()

    return total_loss_epoch/len(train_loader), total_clf_loss/len(train_loader), total_kl_loss/len(train_loader)


# [数据加载逻辑与原脚本一致，仅将 model 指向 GSATSystem]
@torch.no_grad()
def inference_minibatch(loader):
    model.eval() # [cite: 240, 241] 切换到评估模式，GSAT 会自动关闭随机性
    y_preds, y_trues = [], []
    
    for batch in loader:
        batch = batch.to(device)
        # 执行前向传播，此时模型内部 sampler 会执行 (sigmoid(logits) > 0.5) 
        log_probs, _, _ = model(batch.x, batch.edge_index)
        
        # 只保留中心节点的预测结果 [cite: 225]
        y_preds.append(log_probs[:batch.batch_size].argmax(dim=-1).cpu())
        y_trues.append(batch.y[:batch.batch_size].cpu())
        
    y_pred = torch.cat(y_preds, dim=0).numpy().reshape(-1, 1)
    y_true = torch.cat(y_trues, dim=0).numpy().reshape(-1, 1)
    
    return evaluator.eval({'y_true': y_true, 'y_pred': y_pred})['acc']
# ==========================================
# 4. 数据加载与主循环 (GSAT 版)
# ==========================================

import torch.serialization
torch.serialization.add_safe_globals([set, dict, list, tuple])

dataset = PygNodePropPredDataset(name='ogbn-arxiv', root='data', transform=T.ToUndirected())
data = dataset[0]

# [MODIFIED] GSAT 通常不需要显式添加自环，因为其学习的是边重要性概率 p_uv
# 但为了与你的原始流程对齐并保持 GNN 稳定性，我们保留此步骤
edge_index, _ = add_self_loops(data.edge_index, num_nodes=data.num_nodes)
data.edge_index = edge_index

split_idx = dataset.get_idx_split()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 训练用 Loader: 保持采样深度对齐 [20, 15]
train_loader = NeighborLoader(data, num_neighbors=[20, 15], batch_size=Config.batch_size, 
                              input_nodes=split_idx['train'], shuffle=True)

# 推理用 Loader: 看全邻居以保证评估准确性
valid_loader = NeighborLoader(data, num_neighbors=[-1, -1], batch_size=Config.batch_size, 
                              input_nodes=split_idx['valid'], shuffle=False)
test_loader = NeighborLoader(data, num_neighbors=[-1, -1], batch_size=Config.batch_size, 
                             input_nodes=split_idx['test'], shuffle=False)

# 初始化 GSAT 系统
model = GSATSystem(dataset.num_features, Config.hidden_dim, dataset.num_classes).to(device)

# [GSAT MODIFIED] 差分学习率应用
# 采样器学习率通常比主网络低 (0.1x)，以保证信息瓶颈收敛的稳定性 [cite: 645, 651]
sampler_params = list(model.sampler_net.parameters())
other_params = [p for n, p in model.named_parameters() if 'sampler_net' not in n]

optimizer = torch.optim.Adam([
    {'params': other_params, 'lr': Config.lr_init}, 
    {'params': sampler_params, 'lr': Config.lr_init * 0.1}
], weight_decay=Config.weight_decay)

evaluator = Evaluator(name='ogbn-arxiv')
scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[30, 60], gamma=0.5)

# 历史统计容器，新增 kl_loss 以监控信息瓶颈
history = {'train_loss': [], 'clf_loss': [], 'kl_loss': [], 'val_acc': [], 'test_acc': [], 'sparsity': []}
best_valid_acc, stagnant_epochs = 0.0, 0

print(f"Starting GSAT Baseline: Stochastic Attention + IB Regularization (Beta: {Config.beta})...")

for epoch in range(1, Config.epochs + 1):
    # 调用之前定义的 GSAT 专属训练函数
    avg_loss, avg_clf, avg_kl = train_minibatch(epoch)
    scheduler.step()
    
    # 记录损失与当前边保留率 (Sparsity)
    history['train_loss'].append(avg_loss)
    history['clf_loss'].append(avg_clf)
    history['kl_loss'].append(avg_kl)
    
    # 获取当前 Epoch 的平均边保留概率作为 Sparsity 参考
    # 这一步仅作日志记录，不参与计算
    model.eval()
    with torch.no_grad():
        # 这里简单抽取一个 Batch 计算 Sparsity 展示
        sample_batch = next(iter(train_loader)).to(device)
        _, weights, _ = model(sample_batch.x, sample_batch.edge_index)
        avg_sp = weights.mean().item()
        history['sparsity'].append(avg_sp)

    if epoch % 5 == 0: 
        # 调用推理函数 (GSAT 模式下推理会自动关闭 Stochasticity) 
        valid_acc = inference_minibatch(valid_loader)
        test_acc = inference_minibatch(test_loader)
        history['val_acc'].append(valid_acc)
        history['test_acc'].append(test_acc)
        
        if valid_acc > best_valid_acc:
            best_valid_acc = valid_acc
            stagnant_epochs = 0
            torch.save(model.state_dict(), 'best_gsat_arxiv.pt')
            status = "[*] NEW BEST"
        else:
            stagnant_epochs += 1
            status = "[ ]"
            
        print(f"\n{status} Ep: {epoch:03d} | Val:{valid_acc:.4f} | Test:{test_acc:.4f} | KL_L:{avg_kl:.4f} | SP:{avg_sp:.4f}")
        
        if stagnant_epochs >= Config.patience: 
            print(f"\n[Early Stop] Triggered at epoch {epoch}")
            break
        torch.cuda.empty_cache()
    else:
        # 实时打印训练进度
        print(f"      Ep: {epoch:03d} | Clf_L: {avg_clf:.4f} | KL_L: {avg_kl:.4f} | SP: {avg_sp:.4f}", end='\r')

